In [8]:
import pandas as pd
from dotenv import load_dotenv
import os
import anthropic

In [37]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [38]:
qa_data

,knowledge,question,right_answer,hallucinated_answer
0,Arthur's Magazine (1844–1846) was an American ...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,First for Women was started first.
1,The Oberoi family is an Indian family that is ...,The Oberoi family is part of a hotel company t...,Delhi,The Oberoi family's hotel company is based in ...
2,"Allison Beth ""Allie"" Goertz (born March 2, 199...",Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,"Allie Goertz wrote a song about Milhouse, a po..."
3,"Margaret ""Peggy"" Seeger (born June 17, 1935) i...",What nationality was James Henry Miller's wife?,American,James Henry Miller's wife was British.
4,It is a hygroscopic solid that is highly solu...,Cadmium Chloride is slightly soluble in this c...,alcohol,water with a hint of alcohol
...,...,...,...,...
9995,James Norman Hall (22 April 1887 – 5 July 1951...,Are James Norman Hall and Amiri Baraka from th...,yes,James Norman Hall was French.
9996,Love in the Time of Money is a 2002 American r...,The actress who appeared in the 2002 film Love...,1979,The actress who appeared in the 2002 film Love...
9997,"Ape Escape, known in Japan as Excited Saru Get...",how is Ape Escape and Nicktoons Film Festival ...,shorts,Ape Escape and Nicktoons Film Festival are con...
9998,"An accomplished full-forward, Capper kicked 3...",What position did both Warwick Capper and John...,full forward,Warwick Capper played midfield.


In [40]:
df = pd.DataFrame({
    "reference": qa_data.knowledge,
    "query": qa_data.question,
    "response": qa_data.hallucinated_answer
}
)

df = df.head(100)

In [41]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [44]:
import nest_asyncio
import os
from phoenix.evals import HallucinationEvaluator, QAEvaluator, run_evals, AnthropicModel

nest_asyncio.apply()  # This is needed for concurrency in notebook environments

# Get Anthropic API key from environment variable
anthropic_api_key = os.environ.get("ANTHROPIC_API_KEY")

# Set up Claude model for evaluation with API key
eval_model = AnthropicModel(
    model="claude-3-7-sonnet-20250219"
)

# Define your evaluators
hallucination_evaluator = HallucinationEvaluator(eval_model)

# We have to make some minor changes to our dataframe to use the column names expected by our evaluators
# for `hallucination_evaluator` the input df needs to have columns 'output', 'input', 'context'
# for `qa_evaluator` the input df needs to have columns 'output', 'input', 'reference'
df["context"] = df["reference"]
df.rename(columns={"query": "input", "response": "output"}, inplace=True)
assert all(column in df.columns for column in ["output", "input", "context", "reference"])

# Run the evaluators, each evaluator will return a dataframe with evaluation results
# We upload the evaluation results to Phoenix in the next step
hallucination_eval_df = run_evals(
    dataframe=df, evaluators=[hallucination_evaluator], provide_explanation=True
)

run_evals |██████████| 100/100 (100.0%) | ⏳ 01:25<00:00 |  1.17it/s
run_evals |██████████| 100/100 (100.0%) | ⏳ 00:30<00:00 |  2.12it/s

In [45]:
hallucination_eval_df

[           label  score                                        explanation
 0   hallucinated      1  EXPLANATION:\nLet me analyze this step by step...
 1   hallucinated      1  EXPLANATION:\nLet me analyze this step by step...
 2   hallucinated      1  EXPLANATION:\n\nLet me analyze the query, refe...
 3   hallucinated      1  EXPLANATION:\nLet me analyze this step by step...
 4   hallucinated      1  EXPLANATION:\n\nLet me analyze this step by st...
 ..           ...    ...                                                ...
 95  hallucinated      1  EXPLANATION:\nLet me analyze this step by step...
 96  hallucinated      1  EXPLANATION:\n\nLet me analyze this step by st...
 97  hallucinated      1  EXPLANATION:\n\nLet me analyze this step by st...
 98       factual      0  EXPLANATION:\nLet me analyze this step by step...
 99       factual      0  EXPLANATION:\nLet me analyze this step by step...
 
 [100 rows x 3 columns]]